# Modelo de deteccion y priorizacion de fraude en seguros



In [0]:
dbutils.widgets.text("table_name", "workspace.default.muestra_base_fraude", "Tabla fuente")
dbutils.widgets.text("output_schema", "workspace.default", "Schema salida")

TABLE_NAME = dbutils.widgets.get("table_name")
OUTPUT_SCHEMA = dbutils.widgets.get("output_schema")

print(f"Tabla fuente: {TABLE_NAME}")
print(f"Schema salida: {OUTPUT_SCHEMA}")

Tabla fuente: workspace.default.muestra_base_fraude
Schema salida: workspace.default


In [0]:
import re
from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml import Pipeline
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.feature import StringIndexer, VectorAssembler, Imputer, StandardScaler

def clean_name(name: str) -> str:
    name = name.strip()
    name = name.replace("�", "n")
    name = re.sub(r"[^0-9a-zA-Z_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name


raw = spark.table(TABLE_NAME)
rename_map = {c: clean_name(c) for c in raw.columns}
df = raw
for old, new in rename_map.items():
    df = df.withColumnRenamed(old, new)

display(df.limit(5))
print((df.count(), len(df.columns)))

Ramo,Ramo_Desc,Nombre_plan,Codigo_Canal_Comercial_Op,Nombre_Canal_Comercial,Amparo_Desc,Fecha_Primera_Vigencia_Cert,fecha_primera_vigencia_pol,IDENTIFICACION_asegurado,SEXO_asegurado,edad_ingreso_asegurado,edad_actual_asegurado,vigencia_poliza,vigencia_certificado,FEXPEDICION,CODSUC,SUCURSAL,REGIONAL,AGENTE,CODAG,CAUSASTRO,DESCAUSA,DIAGNOSTICO,FSINIESTRO,F_Notificacion,Fecha_Recepcion,Fecha_Apertura,Fecha_Primer_Cierre_Siniestro,Ind_Tipo_Atencion,Ind_Pago_Automatico,Sum_Valor_Reservas_Inicial,Sum_Valor_Reservas,Sum_Valor_Pagos,estado,Cobertura,Tipo_apertura,Fraude_S_N,Periodo_Reporte,Fecha_de_reporte,A_o
083,VIDA DE GRUPO,PLAN VIDA INTEGRAL NO CONTRIBU,CC011,PROMOTORAS,AMPAROS BASICOS SURENTA,2026-01-20T00:00:00.000Z,2026-01-20T00:00:00.000Z,123456,F,35,35,1,1,2026-02-20T00:00:00.000Z,2612,PROMOTORA DE SEGUROS SIGMA,REGIONAL CENTRO,SOCIEDAD DE COBERTURA Y PROTECCION LTDA,20177,01,INFECCIOSAS Y PARASITARIAS,FIEBRE DEL DENGUE [DENGUE CLASICO O HEMORRAGICO],2026-01-02T00:00:00.000Z,2026-03-24T00:00:00.000Z,2026-03-24T00:00:00.000Z,2026-03-24T00:00:00.000Z,24/03/2026,Externa,S,2000000,0,2000000,Tramitado,Renta,Asesor,Fraude,Abril,2026-01-02T00:00:00.000Z,2026
083,VIDA DE GRUPO,PLAN VIDA INTEGRAL NO CONTRIBU,CC011,PROMOTORAS,AMPAROS BASICOS SURENTA,2026-01-20T00:00:00.000Z,2026-01-20T00:00:00.000Z,123456,F,35,35,1,1,2026-02-20T00:00:00.000Z,2612,PROMOTORA DE SEGUROS SIGMA,REGIONAL CENTRO,SOCIEDAD DE COBERTURA Y PROTECCION LTDA,20177,01,INFECCIOSAS Y PARASITARIAS,FIEBRE DEL DENGUE [DENGUE CLASICO O HEMORRAGICO],2026-02-10T00:00:00.000Z,2026-03-24T00:00:00.000Z,2026-03-24T00:00:00.000Z,2026-03-24T00:00:00.000Z,24/03/2026,Externa,S,5000000,0,5000000,Tramitado,Renta,Asesor,Fraude,Abril,2026-02-10T00:00:00.000Z,2026
083,VIDA DE GRUPO,PLAN VIDA INTEGRAL NO CONTRIBU,CC011,PROMOTORAS,AMPAROS BASICOS SURENTA,2026-01-20T00:00:00.000Z,2026-01-20T00:00:00.000Z,123456,F,35,35,1,1,2026-02-20T00:00:00.000Z,2612,PROMOTORA DE SEGUROS SIGMA,REGIONAL CENTRO,SOCIEDAD DE COBERTURA Y PROTECCION LTDA,20177,01,INFECCIOSAS Y PARASITARIAS,FIEBRE DEL DENGUE [DENGUE CLASICO O HEMORRAGICO],2025-10-04T00:00:00.000Z,2026-03-24T00:00:00.000Z,2026-03-24T00:00:00.000Z,2026-03-24T00:00:00.000Z,24/03/2026,Externa,S,3200000,0,3200000,Tramitado,Renta,Asesor,Fraude,Abril,2025-10-04T00:00:00.000Z,2026
BAN,BANCASEGUROS,RESPALDO DE VIDA,CC014,ALIANZA TUYA,RENTA MENSUAL HOSPITALARIA,2016-07-04T00:00:00.000Z,2016-07-04T00:00:00.000Z,123457,M,60,66,3,3,2016-07-04T00:00:00.000Z,2797,TARJETA ALKOSTO,OFICINA CENTRAL,SEGUROS GENERALES SURAMERICANA S.A.,4999,19,"TRAUMATISMOS, ENVENENAMIENTO",FRACTURA DEL ESTERNÓN,2017-01-01T00:00:00.000Z,2022-06-30T00:00:00.000Z,2022-06-30T00:00:00.000Z,2022-06-30T00:00:00.000Z,08/07/2022,Cliente Servidor,N,700000,2100,0,Objetado,Renta,Reclamaciones Digital,Fraude,Julio,2022-07-08T14:36:00.000Z,2022
081,BANCASEGUROS,RESPALDO DE VIDA,CC014,ALIANZA TUYA,RENTA MENSUAL HOSPITALARIA,2016-07-04T00:00:00.000Z,2016-07-04T00:00:00.000Z,123457,M,60,66,3,3,2016-07-04T00:00:00.000Z,2797,TARJETA ALKOSTO,OFICINA CENTRAL,SEGUROS GENERALES SURAMERICANA S.A.,4999,19,"TRAUMATISMOS, ENVENENAMIENTO",FRACTURA DE COSTILLA,2017-01-26T00:00:00.000Z,2022-06-30T00:00:00.000Z,2022-06-30T00:00:00.000Z,2022-06-30T00:00:00.000Z,08/07/2022,Cliente Servidor,N,700000,2100,0,Objetado,Renta,Reclamaciones Digital,Fraude,Julio,2022-07-08T14:36:00.000Z,2022


(12776, 40)


## 1. Calidad de datos y variable objetivo

In [0]:
target_col = "Fraude_S_N"
if target_col not in df.columns:
    raise ValueError(f"No encontre la columna objetivo {target_col}. Columnas: {df.columns}")

df = df.withColumn(
    "label",
    F.when(F.lower(F.trim(F.col(target_col))) == F.lit("fraude"), F.lit(1.0)).otherwise(F.lit(0.0)),
)

display(df.groupBy(target_col, "label").count())

quality = []
for c in df.columns:
    quality.append(
        df.select(
            F.lit(c).alias("columna"),
            F.count(F.when(F.col(c).isNull(), c)).alias("nulos"),
            F.approx_count_distinct(F.col(c)).alias("cardinalidad_aprox"),
        )
    )

quality_df = quality[0]
for q in quality[1:]:
    quality_df = quality_df.unionByName(q)

display(quality_df.orderBy(F.desc("nulos"), F.desc("cardinalidad_aprox")))

Fraude_S_N,label,count
Fraude,1.0,7832
No es fraude,0.0,4944


columna,nulos,cardinalidad_aprox
Amparo_Desc,7,52
IDENTIFICACION_asegurado,0,3418
Sum_Valor_Reservas_Inicial,0,3303
Fecha_de_reporte,0,2563
FSINIESTRO,0,2507
Sum_Valor_Reservas,0,1964
Sum_Valor_Pagos,0,1931
Fecha_Primera_Vigencia_Cert,0,1923
fecha_primera_vigencia_pol,0,1880
FEXPEDICION,0,1825


## 2. Feature engineering y control de leakage

Se excluyen identificadores y variables posteriores al momento de scoring. Las fechas se transforman en diferencias de dias y variables calendario.

In [0]:
date_cols = [
    "Fecha_Primera_Vigencia_Cert",
    "fecha_primera_vigencia_pol",
    "FEXPEDICION",
    "FSINIESTRO",
    "F_Notificacion",
    "Fecha_Recepcion",
    "Fecha_Apertura",
]

for c in date_cols:
    if c in df.columns:
        df = df.withColumn(c, F.to_date(F.col(c)))

if {"FSINIESTRO", "Fecha_Apertura"}.issubset(set(df.columns)):
    df = df.withColumn("dias_siniestro_a_apertura", F.datediff("Fecha_Apertura", "FSINIESTRO"))
if {"FSINIESTRO", "F_Notificacion"}.issubset(set(df.columns)):
    df = df.withColumn("dias_siniestro_a_notificacion", F.datediff("F_Notificacion", "FSINIESTRO"))
if {"F_Notificacion", "Fecha_Apertura"}.issubset(set(df.columns)):
    df = df.withColumn("dias_notificacion_a_apertura", F.datediff("Fecha_Apertura", "F_Notificacion"))
if {"Fecha_Primera_Vigencia_Cert", "FSINIESTRO"}.issubset(set(df.columns)):
    df = df.withColumn("dias_vigencia_cert_a_siniestro", F.datediff("FSINIESTRO", "Fecha_Primera_Vigencia_Cert"))
if {"fecha_primera_vigencia_pol", "FSINIESTRO"}.issubset(set(df.columns)):
    df = df.withColumn("dias_vigencia_pol_a_siniestro", F.datediff("FSINIESTRO", "fecha_primera_vigencia_pol"))
if {"FEXPEDICION", "FSINIESTRO"}.issubset(set(df.columns)):
    df = df.withColumn("dias_expedicion_a_siniestro", F.datediff("FSINIESTRO", "FEXPEDICION"))
if "FSINIESTRO" in df.columns:
    df = df.withColumn("mes_siniestro", F.month("FSINIESTRO").cast("string"))
    df = df.withColumn("dia_semana_siniestro", F.dayofweek("FSINIESTRO").cast("string"))
if {"edad_actual_asegurado", "edad_ingreso_asegurado"}.issubset(set(df.columns)):
    df = df.withColumn("delta_edad", F.col("edad_actual_asegurado") - F.col("edad_ingreso_asegurado"))
    df = df.withColumn("edad_actual_invalida", F.when(F.col("edad_actual_asegurado") < 0, 1.0).otherwise(0.0))
if "Sum_Valor_Reservas_Inicial" in df.columns:
    df = df.withColumn("log_reserva_inicial", F.log1p(F.greatest(F.col("Sum_Valor_Reservas_Inicial"), F.lit(0))))
    df = df.withColumn("reserva_inicial_es_cero", F.when(F.col("Sum_Valor_Reservas_Inicial") == 0, 1.0).otherwise(0.0))

strict_exclude = {
    target_col,
    "label",
    "IDENTIFICACION_asegurado",
    "Fecha_Primer_Cierre_Siniestro",
    "estado",
    "Periodo_Reporte",
    "Fecha_de_reporte",
    "Ano",
    "An_o",
    "Sum_Valor_Reservas",
    "Sum_Valor_Pagos",
    *date_cols,
}

high_cardinality_exclude = {
    "AGENTE",
    "DIAGNOSTICO",
    "SUCURSAL",
    "Nombre_plan",
    "Amparo_Desc",
}

feature_cols = [
    c for c in df.columns
    if c not in strict_exclude and c not in high_cardinality_exclude
]
display(spark.createDataFrame([(c,) for c in feature_cols], ["variables_modelo"]))

variables_modelo
Ramo
Ramo_Desc
Codigo_Canal_Comercial_Op
Nombre_Canal_Comercial
SEXO_asegurado
edad_ingreso_asegurado
edad_actual_asegurado
vigencia_poliza
vigencia_certificado
CODSUC


## 3. Split temporal

In [0]:
order_col = "Fecha_Apertura" if "Fecha_Apertura" in df.columns else "FSINIESTRO"
w = Window.orderBy(F.col(order_col).asc_nulls_last())
df_split = df.withColumn("rn", F.row_number().over(w))
n = df_split.count()
split_at = int(n * 0.75)

train_df = df_split.filter(F.col("rn") <= split_at).drop("rn")
test_df = df_split.filter(F.col("rn") > split_at).drop("rn")

print(f"Train: {train_df.count()} | Test: {test_df.count()}")
display(train_df.groupBy("label").count())
display(test_df.groupBy("label").count())

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Train: 9582 | Test: 3194


label,count
0.0,2813
1.0,6769


label,count
0.0,2131
1.0,1063


## 4. Preprocesamiento y tres modelos

In [0]:
numeric_cols = []

for name, dtype in train_df.select(feature_cols).dtypes:
    if dtype in {"int", "bigint", "double", "float", "decimal", "smallint", "tinyint"}:
        numeric_cols.append(name)

# Version liviana para Databricks Free Edition:
# entrenamos solo con variables numericas y temporales derivadas.
categorical_cols = []

print("Variables numericas usadas:")
print(numeric_cols)

imputed_numeric = [f"{c}_imputed" for c in numeric_cols]

imputer = Imputer(
    inputCols=numeric_cols,
    outputCols=imputed_numeric
).setStrategy("median")

assembler = VectorAssembler(
    inputCols=imputed_numeric,
    outputCol="features_raw",
    handleInvalid="keep"
)

scaler = StandardScaler(
    inputCol="features_raw",
    outputCol="features"
)

models = {
    "logistic_regression": LogisticRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=30,
        regParam=0.10,
        elasticNetParam=0.0
    ),
    "random_forest": RandomForestClassifier(
        featuresCol="features",
        labelCol="label",
        numTrees=50,
        maxDepth=6,
        seed=42
    ),
    "gradient_boosting": GBTClassifier(
        featuresCol="features",
        labelCol="label",
        maxIter=30,
        maxDepth=3,
        seed=42
    ),
}

pipelines = {
    name: Pipeline(stages=[imputer, assembler, scaler, model])
    for name, model in models.items()
}

Variables numericas usadas:
['edad_ingreso_asegurado', 'edad_actual_asegurado', 'vigencia_poliza', 'vigencia_certificado', 'Sum_Valor_Reservas_Inicial', 'A_o', 'dias_siniestro_a_apertura', 'dias_siniestro_a_notificacion', 'dias_notificacion_a_apertura', 'dias_vigencia_cert_a_siniestro', 'dias_vigencia_pol_a_siniestro', 'dias_expedicion_a_siniestro', 'delta_edad', 'edad_actual_invalida', 'log_reserva_inicial', 'reserva_inicial_es_cero']


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
APPROVED_FEATURES = [
    "edad_ingreso_asegurado",
    "edad_actual_asegurado",
    "vigencia_poliza",
    "vigencia_certificado",
    "Sum_Valor_Reservas_Inicial",
    "A_o",
    "dias_siniestro_a_apertura",
    "dias_siniestro_a_notificacion",
    "dias_notificacion_a_apertura",
    "dias_vigencia_cert_a_siniestro",
    "dias_vigencia_pol_a_siniestro",
    "dias_expedicion_a_siniestro",
    "delta_edad",
    "edad_actual_invalida",
    "log_reserva_inicial",
    "reserva_inicial_es_cero",
]

missing_features = [
    column
    for column in APPROVED_FEATURES
    if column not in train_df.columns
]

unexpected_features = [
    column
    for column in numeric_cols
    if column not in APPROVED_FEATURES
]

if missing_features:
    raise ValueError(
        f"Faltan variables obligatorias del modelo: {missing_features}"
    )

if unexpected_features:
    raise ValueError(
        f"El pipeline generó variables no aprobadas: {unexpected_features}"
    )

if numeric_cols != APPROVED_FEATURES:
    raise ValueError(
        "El orden de las variables no coincide con el contrato aprobado"
    )

print("Contrato del modelo aprobado correctamente")
train_df.select(APPROVED_FEATURES).printSchema()

Contrato del modelo aprobado correctamente
root
 |-- edad_ingreso_asegurado: long (nullable = true)
 |-- edad_actual_asegurado: long (nullable = true)
 |-- vigencia_poliza: long (nullable = true)
 |-- vigencia_certificado: long (nullable = true)
 |-- Sum_Valor_Reservas_Inicial: long (nullable = true)
 |-- A_o: long (nullable = true)
 |-- dias_siniestro_a_apertura: integer (nullable = true)
 |-- dias_siniestro_a_notificacion: integer (nullable = true)
 |-- dias_notificacion_a_apertura: integer (nullable = true)
 |-- dias_vigencia_cert_a_siniestro: integer (nullable = true)
 |-- dias_vigencia_pol_a_siniestro: integer (nullable = true)
 |-- dias_expedicion_a_siniestro: integer (nullable = true)
 |-- delta_edad: long (nullable = true)
 |-- edad_actual_invalida: double (nullable = false)
 |-- log_reserva_inicial: double (nullable = true)
 |-- reserva_inicial_es_cero: double (nullable = false)



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
import inspect
import mlflow
import mlflow.spark

print("Versión de MLflow:", mlflow.__version__)
print("Firma de mlflow.spark.log_model:")
print(inspect.signature(mlflow.spark.log_model))

Versión de MLflow: 3.8.1
Firma de mlflow.spark.log_model:
(spark_model, artifact_path, conda_env=None, code_paths=None, dfs_tmpdir=None, registered_model_name=None, signature: mlflow.models.signature.ModelSignature = None, input_example: Union[pandas.core.frame.DataFrame, numpy.ndarray, dict, list, ForwardRef('csr_matrix'), ForwardRef('csc_matrix'), str, bytes, tuple] = None, await_registration_for=300, pip_requirements=None, extra_pip_requirements=None, metadata=None)


In [0]:
evaluator_roc = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

evaluator_pr = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

results = []
best_model = None
best_model_name = None
best_pr = -1.0

for name, pipeline in pipelines.items():
    print(f"Entrenando {name}...")
    fitted_model = pipeline.fit(train_df)
    pred = fitted_model.transform(test_df)
    pred = pred.withColumn("score_fraude", vector_to_array(F.col("probability"))[1])

    roc = evaluator_roc.evaluate(pred)
    pr = evaluator_pr.evaluate(pred)

    pred_label = pred.withColumn(
        "pred_label",
        F.when(F.col("score_fraude") >= 0.5, 1.0).otherwise(0.0)
    )

    cm = pred_label.groupBy("label", "pred_label").count()
    display(cm.withColumn("model", F.lit(name)))

    results.append((name, roc, pr))

    if pr > best_pr:
        best_pr = pr
        best_model_name = name
        best_model = fitted_model

metrics_df = spark.createDataFrame(
    results,
    ["model", "roc_auc", "area_under_pr"]
).orderBy(F.desc("area_under_pr"))

display(metrics_df)

print(f"Mejor modelo: {best_model_name}")

Entrenando logistic_regression...


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


label,pred_label,count,model
0.0,0.0,2084,logistic_regression
1.0,0.0,1031,logistic_regression
1.0,1.0,32,logistic_regression
0.0,1.0,47,logistic_regression


Entrenando random_forest...


label,pred_label,count,model
0.0,0.0,2072,random_forest
1.0,0.0,919,random_forest
1.0,1.0,144,random_forest
0.0,1.0,59,random_forest


Entrenando gradient_boosting...


label,pred_label,count,model
0.0,0.0,2067,gradient_boosting
1.0,0.0,968,gradient_boosting
1.0,1.0,95,gradient_boosting
0.0,1.0,64,gradient_boosting


model,roc_auc,area_under_pr
random_forest,0.7850787527927352,0.6127730517022864
gradient_boosting,0.7058185112214841,0.5229502001832373
logistic_regression,0.2938194983076946,0.25965308038356977


Mejor modelo: random_forest


In [0]:
print("best_model existe:", "best_model" in globals())
print("Mejor modelo aprobado:", best_model_name if "best_model_name" in globals() else "No disponible")

best_model existe: True
Mejor modelo aprobado: random_forest


## 5. Lift por deciles y segmentos de riesgo

In [0]:
scored = best_model.transform(test_df)
scored = scored.withColumn("score_fraude", vector_to_array(F.col("probability"))[1])

w_score = Window.orderBy(F.desc("score_fraude"))
scored = scored.withColumn("rank_score", F.row_number().over(w_score))
scored = scored.withColumn("decil", F.ntile(10).over(w_score))

base_rate = scored.agg(F.avg("label").alias("base_rate")).first()["base_rate"]

lift = (
    scored.groupBy("decil")
    .agg(
        F.count("*").alias("casos"),
        F.sum("label").alias("fraudes"),
        F.min("score_fraude").alias("score_min"),
        F.max("score_fraude").alias("score_max"),
        F.avg("label").alias("tasa_fraude"),
    )
    .withColumn("lift", F.col("tasa_fraude") / F.lit(base_rate))
    .orderBy("decil")
)

display(lift)

segment_cols = [
    "Ramo_Desc",
    "Nombre_Canal_Comercial",
    "REGIONAL",
    "SUCURSAL",
    "Cobertura",
    "Tipo_apertura",
    "SEXO_asegurado",
    "Ind_Pago_Automatico",
]

for c in segment_cols:
    if c in scored.columns:
        display(
            scored.groupBy(c)
            .agg(F.count("*").alias("casos"), F.avg("label").alias("tasa_fraude"), F.avg("score_fraude").alias("score_promedio"))
            .filter(F.col("casos") >= 30)
            .orderBy(F.desc("tasa_fraude"))
            .limit(10)
        )

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


decil,casos,fraudes,score_min,score_max,tasa_fraude,lift
1,320,218.0,0.4663773253001871,0.7851136076095264,0.68125,2.0469543744120413
2,320,226.0,0.4296664901149328,0.4660730540464438,0.70625,2.1220719661335843
3,320,175.0,0.403032389950828,0.42965216092021685,0.546875,1.6431973189087488
4,320,109.0,0.38046251630548705,0.40293653580006,0.340625,1.0234771872060207
5,319,106.0,0.36090960301151426,0.38046104060925434,0.3322884012539185,0.9984281783678417
6,319,79.0,0.3374609903217501,0.3605172667508509,0.2476489028213166,0.7441115668967876
7,319,57.0,0.31054366088592084,0.3374128224034284,0.1786833855799373,0.5368906242166696
8,319,34.0,0.2815848119267772,0.31041364512809316,0.10658307210031348,0.3202505477783643
9,319,44.0,0.2449605051946288,0.2815712860908535,0.13793103448275862,0.41444188536023613
10,319,15.0,0.10267942430270725,0.2449605051946288,0.047021943573667714,0.14128700637280778


Ramo_Desc,casos,tasa_fraude,score_promedio
VIDA DE GRUPO,1247,0.45148356054530875,0.3744372227896849
BANCASEGUROS,701,0.29814550641940085,0.3590577573826816
VIDA INDIVIDUAL,1199,0.23519599666388658,0.34393629299682493
ACCIDENTES PERSONALES,41,0.21951219512195122,0.35951784694761035


Nombre_Canal_Comercial,casos,tasa_fraude,score_promedio
ALIANZA EXITO,110,0.990909090909091,0.4013625757235506
PROMOTORAS,1281,0.4496487119437939,0.38076212549174976
BANCOLOMBIA DEUDORES,154,0.33766233766233766,0.354476965905798
CORREDORES,154,0.2922077922077922,0.32924244950185355
GRAN EMPRESA,64,0.203125,0.32199186252767165
SUCURSALES,801,0.19850187265917604,0.33540308466470103
OTROS BANCOLOMBIA VOLUNTARIOS,87,0.1724137931034483,0.3415742402788647
BANCOLOMBIA VOLUNTARIOS,471,0.14861995753715498,0.35197437122944736


REGIONAL,casos,tasa_fraude,score_promedio
REGIONAL CENTRO,670,0.7059701492537314,0.41847918450204186
REGIONAL ANTIOQUIA,734,0.29155313351498635,0.34403188625966946
REGIONAL EJE CAFETERO,169,0.28994082840236685,0.34634814278056164
REGIONAL OCCIDENTE,306,0.28431372549019607,0.33863365679236784
OFICINA CENTRAL,763,0.19397116644823068,0.35070371270001377
REGIONAL NORTE,550,0.16727272727272727,0.3359497996432631


SUCURSAL,casos,tasa_fraude,score_promedio
PROMOTORA DE SEGUROS SIGMA,376,1.0,0.46519368486824925
SEGUROS EXITO BELLO,56,1.0,0.4021559519628159
PROMOTORA GESTIONARTE,53,0.6037735849056604,0.38976123823634395
BRILLA FASE 0,96,0.46875,0.3511154455592644
SUCURSAL SAN FERNANDO 5,51,0.3137254901960784,0.36103480141392363
SUCURSAL SAN FERNANDO 4,49,0.2857142857142857,0.34096336310145625
SUCURSAL CALI 2,78,0.28205128205128205,0.327961787943147
SUCURSAL CALI 1,32,0.28125,0.3408274562792823
BANCOLOMBIA TMK BANCA,57,0.24561403508771928,0.33856019959594524
BANCOLOMBIA DEUDORES CONSUMO Y OTROS,89,0.19101123595505617,0.3456156475929449


Cobertura,casos,tasa_fraude,score_promedio
Invalidez,418,0.5023923444976076,0.37631063520040664
Renta,1975,0.34430379746835443,0.3566973464875205
Ap Complementario,167,0.23353293413173654,0.3729663651932134
Vida,634,0.2113564668769716,0.35368972666536236


Tipo_apertura,casos,tasa_fraude,score_promedio
Asesor,960,0.49583333333333335,0.38704833898999474
Reclamaciones Digital,1305,0.2636015325670498,0.34900479859756106
Linea,902,0.26053215077605324,0.34496969783407655


SEXO_asegurado,casos,tasa_fraude,score_promedio
F,1116,0.4032258064516129,0.36315601958495713
M,2078,0.29499518768046196,0.3575638386250802


Ind_Pago_Automatico,casos,tasa_fraude,score_promedio
S,731,0.5567715458276333,0.3950686464053805
N,2463,0.26634185952090944,0.3489665424268752


In [0]:
spark.sql("""
CREATE VOLUME IF NOT EXISTS workspace.fraude_prod.mlflow_tmp
COMMENT 'Almacenamiento temporal para registrar modelos Spark con MLflow'
""")

MLFLOW_TMP_PATH = (
    "/Volumes/workspace/fraude_prod/"
    "mlflow_tmp/spark_models"
)

dbutils.fs.mkdirs(MLFLOW_TMP_PATH)

print("Volumen temporal preparado:")
print(MLFLOW_TMP_PATH)

Volumen temporal preparado:
/Volumes/workspace/fraude_prod/mlflow_tmp/spark_models


In [0]:
from pyspark.sql import functions as F
from mlflow.models import infer_signature

MODEL_NAME = (
    "workspace.fraude_prod."
    "fraude_random_forest"
)

# Estandarizamos todas las entradas numéricas como double.
input_example_spark = (
    train_df
    .select([
        F.col(column).cast("double").alias(column)
        for column in APPROVED_FEATURES
    ])
    .limit(10)
)

output_example_spark = (
    best_model
    .transform(input_example_spark)
    .select(
        F.col("prediction")
        .cast("double")
        .alias("prediction")
    )
)

input_example = input_example_spark.toPandas()
output_example = output_example_spark.toPandas()

model_signature = infer_signature(
    input_example,
    output_example
)

print("Modelo a registrar:", MODEL_NAME)
print("Columnas de entrada:", list(input_example.columns))
print("Firma creada correctamente:")
print(model_signature)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Modelo a registrar: workspace.fraude_prod.fraude_random_forest
Columnas de entrada: ['edad_ingreso_asegurado', 'edad_actual_asegurado', 'vigencia_poliza', 'vigencia_certificado', 'Sum_Valor_Reservas_Inicial', 'A_o', 'dias_siniestro_a_apertura', 'dias_siniestro_a_notificacion', 'dias_notificacion_a_apertura', 'dias_vigencia_cert_a_siniestro', 'dias_vigencia_pol_a_siniestro', 'dias_expedicion_a_siniestro', 'delta_edad', 'edad_actual_invalida', 'log_reserva_inicial', 'reserva_inicial_es_cero']
Firma creada correctamente:
inputs: 
  ['edad_ingreso_asegurado': double (required), 'edad_actual_asegurado': double (required), 'vigencia_poliza': double (required), 'vigencia_certificado': double (required), 'Sum_Valor_Reservas_Inicial': double (required), 'A_o': double (required), 'dias_siniestro_a_apertura': double (required), 'dias_siniestro_a_notificacion': double (required), 'dias_notificacion_a_apertura': double (required), 'dias_vigencia_cert_a_siniestro': double (required), 'dias_vigencia_

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
import mlflow
import mlflow.spark

mlflow.set_registry_uri("databricks-uc")

with mlflow.start_run(
    run_name=(
        "registro_fraude_random_"
        "forest_aprobado_v1"
    )
) as run:

    mlflow.set_tags({
        "estado_validacion": "aprobado",
        "caso_uso": (
            "priorizacion_fraude_seguros"
        ),
        "entorno_objetivo": "produccion",
        "tipo_scoring": "batch_y_online",
        "algoritmo": "random_forest",
    })

    mlflow.log_params({
        "num_trees": 50,
        "max_depth": 6,
        "seed": 42,
        "numero_variables": len(
            APPROVED_FEATURES
        ),
    })

    mlflow.log_metrics({
        "roc_auc_aprobado":
            0.7850787527927352,
        "area_under_pr_aprobado":
            0.6127730517022864,
        "captura_fraude_top_20_pct":
            0.418,
    })

    model_info = mlflow.spark.log_model(
        spark_model=best_model,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
        signature=model_signature,
        input_example=input_example,
        metadata={
            "estado": "aprobado",
            "finalidad": (
                "priorizacion_de_reclamaciones"
            ),
            "advertencia": (
                "El score prioriza investigacion "
                "y no confirma fraude"
            ),
        },
        dfs_tmpdir=MLFLOW_TMP_PATH,
        await_registration_for=300,
    )

    RUN_ID = run.info.run_id

print("Modelo registrado correctamente")
print("Run ID:", RUN_ID)
print("URI:", model_info.model_uri)

2026/09/19 02:29:58 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.1.0+databricks.connect.18.1.9) contains a local version label (+databricks.connect.18.1.9). MLflow logged a pip requirement for this package as 'pyspark==4.1.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/09/19 02:30:03 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: /local_disk0/user_tmp_data/spark-c8814ed0-b730-4770-b2d5-b6/tmp06l5fjyp/model, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 
2026/09/19 02:30:03 WARNING mlflow.models.model: Failed to validate serving input example {
  "dataframe_split": {
    "columns": [
      "e.... Alternatively, you can avoid passing input example and pass model signature instead when logging the model. To ens

Uploading artifacts:   0%|          | 0/46 [00:00<?, ?it/s]

Modelo registrado correctamente
Run ID: 5cac913910464058a36a6c466dce8008
URI: runs:/5cac913910464058a36a6c466dce8008/model


🔗 Created version '1' of model 'workspace.fraude_prod.fraude_random_forest': https://<workspace-databricks>/explore/data/models/workspace/fraude_prod/fraude_random_forest/version/1?o=<workspace-id>


In [0]:
import os
import mlflow

MLFLOW_TMP_PATH = (
    "/Volumes/workspace/fraude_prod/"
    "mlflow_tmp/spark_models"
)

os.environ["MLFLOW_DFS_TMP"] = MLFLOW_TMP_PATH

mlflow.set_registry_uri("databricks-uc")

print(
    "MLFLOW_DFS_TMP:",
    os.environ["MLFLOW_DFS_TMP"]
)

MLFLOW_DFS_TMP: /Volumes/workspace/fraude_prod/mlflow_tmp/spark_models


In [0]:
from mlflow import MlflowClient

MODEL_NAME = (
    "workspace.fraude_prod."
    "fraude_random_forest"
)

client = MlflowClient()

versions = list(
    client.search_model_versions(
        f"name='{MODEL_NAME}'"
    )
)

print("Número de versiones:", len(versions))

for version in versions:
    print(
        "Versión:",
        version.version,
        "| Estado:",
        version.status,
        "| Run ID:",
        version.run_id
    )

Número de versiones: 1
Versión: 1 | Estado: READY | Run ID: 5cac913910464058a36a6c466dce8008


In [0]:
versions = list(
    client.search_model_versions(
        f"name='{MODEL_NAME}'"
    )
)

ready_versions = [
    version
    for version in versions
    if version.status == "READY"
]

if not ready_versions:
    raise ValueError(
        "No existe una versión READY"
    )

latest_version = max(
    ready_versions,
    key=lambda item: int(item.version)
)

MODEL_VERSION = latest_version.version

client.set_registered_model_alias(
    name=MODEL_NAME,
    alias="Champion",
    version=MODEL_VERSION,
)

client.set_model_version_tag(
    name=MODEL_NAME,
    version=MODEL_VERSION,
    key="estado_validacion",
    value="aprobado",
)

client.set_model_version_tag(
    name=MODEL_NAME,
    version=MODEL_VERSION,
    key="uso_autorizado",
    value="scoring_productivo",
)

print("Modelo:", MODEL_NAME)
print("Versión:", MODEL_VERSION)
print("Alias: Champion")
print(
    "URI productiva:",
    f"models:/{MODEL_NAME}@Champion"
)

Modelo: workspace.fraude_prod.fraude_random_forest
Versión: 1
Alias: Champion
URI productiva: models:/workspace.fraude_prod.fraude_random_forest@Champion


In [0]:
import mlflow
import mlflow.spark

from pyspark.ml.functions import vector_to_array
from pyspark.sql import functions as F

MODEL_URI = (
    "models:/workspace.fraude_prod."
    "fraude_random_forest@Champion"
)

production_model = mlflow.spark.load_model(
    MODEL_URI,
    dfs_tmpdir=MLFLOW_TMP_PATH,
)

print("Modelo Champion cargado correctamente")

Modelo Champion cargado correctamente


In [0]:
validation_sample = (
    test_df
    .select([
        F.col(column)
        .cast("double")
        .alias(column)
        for column in APPROVED_FEATURES
    ])
    .limit(100)
    .withColumn(
        "_validation_id",
        F.monotonically_increasing_id()
    )
)

original_scores = (
    best_model
    .transform(validation_sample)
    .withColumn(
        "score_original",
        vector_to_array(
            F.col("probability")
        )[1]
    )
    .select(
        "_validation_id",
        "score_original"
    )
)

registered_scores = (
    production_model
    .transform(validation_sample)
    .withColumn(
        "score_registrado",
        vector_to_array(
            F.col("probability")
        )[1]
    )
    .select(
        "_validation_id",
        "score_registrado"
    )
)

comparison = (
    original_scores
    .join(
        registered_scores,
        "_validation_id"
    )
    .withColumn(
        "diferencia",
        F.abs(
            F.col("score_original")
            - F.col("score_registrado")
        )
    )
)

max_difference = (
    comparison
    .agg(
        F.max("diferencia")
        .alias("max_difference")
    )
    .first()["max_difference"]
)

display(
    comparison.orderBy(
        F.desc("diferencia")
    )
)

print("Diferencia máxima:", max_difference)

if max_difference is None:
    raise ValueError(
        "No se generaron comparaciones"
    )

if max_difference > 1e-12:
    raise ValueError(
        "El modelo registrado no reproduce "
        "los scores originales"
    )

print(
    "Validación exitosa: Champion reproduce "
    "exactamente el modelo aprobado"
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


_validation_id,score_original,score_registrado,diferencia
0,0.3183426372324932,0.3183426372324932,0.0
1,0.4152581095705098,0.4152581095705098,0.0
2,0.37175003245493626,0.37175003245493626,0.0
3,0.4062182886701198,0.4062182886701198,0.0
4,0.2169707906203783,0.2169707906203783,0.0
5,0.3102607584586063,0.3102607584586063,0.0
6,0.3829666385889153,0.3829666385889153,0.0
7,0.35254209624872385,0.35254209624872385,0.0
8,0.6132568714516498,0.6132568714516498,0.0
9,0.4749864555475397,0.4749864555475397,0.0


Diferencia máxima: 0.0
Validación exitosa: Champion reproduce exactamente el modelo aprobado


## 6. Guardar resultados en Delta

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {OUTPUT_SCHEMA}")

metrics_df.write.mode("overwrite").saveAsTable(f"{OUTPUT_SCHEMA}.fraude_model_metrics")
lift.write.mode("overwrite").saveAsTable(f"{OUTPUT_SCHEMA}.fraude_decile_lift")

scored.select(
    *[c for c in scored.columns if c in ["label", "score_fraude", "decil", "Ramo_Desc", "Nombre_Canal_Comercial", "REGIONAL", "SUCURSAL", "Cobertura", "Tipo_apertura"]],
).write.mode("overwrite").saveAsTable(f"{OUTPUT_SCHEMA}.fraude_scored_test")

print("Tablas guardadas:")
print(f"- {OUTPUT_SCHEMA}.fraude_model_metrics")
print(f"- {OUTPUT_SCHEMA}.fraude_decile_lift")
print(f"- {OUTPUT_SCHEMA}.fraude_scored_test")

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Tablas guardadas:
- workspace.default.fraude_model_metrics
- workspace.default.fraude_decile_lift
- workspace.default.fraude_scored_test


## 7. Lectura de negocio

- El score permite priorizar los casos con mayor probabilidad de fraude.
- El top decil concentra una tasa de fraude superior a la tasa base si el modelo esta ordenando correctamente.
- La salida recomendada para negocio es: identificador operativo, score, decil, prioridad, fecha de scoring y version del modelo.
- El modelo no prueba fraude: focaliza investigacion.